# Graph Conv Nets for similarity prediction

In [108]:
import os
import glob
import torch
# from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import HeteroData, InMemoryDataset
import torch_geometric.transforms as T
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset, HeteroData

import random

In [109]:
random.seed(42)
torch.manual_seed(42)

In [160]:
def parse_cnf(file_path):
	"""
	Parses a DIMACS CNF file and extracts variable and clause mappings.
	
	Args:
		file_path (str): The path to the .cnf file.
		
	Returns:
		tuple: (num_vars, num_clauses, pos_literals, neg_literals)
			- num_vars (int): Total number of variables.
			- num_clauses (int): Total number of clauses.
			- pos_literals (tuple of lists): (A, B) where variable A[i] appears positively in clause B[i].
			- neg_literals (tuple of lists): (C, D) where variable C[i] appears negatively in clause D[i].
	"""
	num_vars = 0
	num_clauses_expected = 0
	
	# A corresponds to the variable index, B corresponds to the clause index
	A_pos, B_pos = [], []
	A_neg, B_neg = [], []
	
	current_clause_idx = 0
	
	with open(file_path, 'r') as file:
		for line in file:
			line = line.strip()
			
			# Skip comments and empty lines
			if not line or line.startswith('c'):
				continue
				
			# Parse the problem definition line
			if line.startswith('p cnf'):
				parts = line.split()
				# format is usually: p cnf <vars> <clauses>
				num_vars = int(parts[2])
				num_clauses_expected = int(parts[3])
				continue
				
			# Parse the literals
			# Split by whitespace to handle arbitrary spacing
			parts = line.split()
			
			for part in parts:
				# Some CNF files have a '%' to indicate the end of the data stream
				if part == '%':
					break
					
				val = int(part)
				
				if val == 0:
					# A '0' indicates the end of the current clause
					current_clause_idx += 1
				elif val > 0:
					# Positive literal
					A_pos.append(val-1)
					B_pos.append(current_clause_idx)
				else:
					# Negative literal (we store the absolute variable number)
					A_neg.append(abs(val)-1)
					B_neg.append(current_clause_idx)
					
	# Use the actual counted clauses as a fallback, though it should match num_clauses_expected
	actual_num_clauses = current_clause_idx
	
	return num_vars, actual_num_clauses, (A_pos, B_pos), (A_neg, B_neg)

# --- Example Usage ---
# num_vars, num_clauses, pos_lits, neg_lits = parse_cnf('example.cnf')
# A_pos, B_pos = pos_lits
# A_neg, B_neg = neg_lits

In [161]:
from setup import cnf_path

vars, clauses, pos_lits, neg_lits = parse_cnf(cnf_path/'miter_mult_14bit.cnf')

In [162]:
class TripletCNFDataset(Dataset):
	def __init__(self, file_class_mapping, num_bw, hidden_channels=32, transform=None):
		"""
		Args:
			file_class_mapping (list of tuples): [('path/to/file1.cnf', 'class_1'), ...]
			hidden_channels (int): Feature dimension size.
		"""
		super().__init__(root=None, transform=transform)
		self.hidden_channels = hidden_channels
		self.pre_transform = T.ToUndirected()
		self.num_bw = num_bw
		# We will store parsed graphs grouped by their class label
		self.graphs_by_class = {}
		self.all_graphs = [] # Keep a flat list just to track total dataset size
		
		self.lit_emb = torch.randn(1, self.hidden_channels).requires_grad_(True)
		self.cls_emb = torch.randn(1, self.hidden_channels).requires_grad_(True)

		print("Processing CNF files into memory...")
		for file_path, label in file_class_mapping:
			num_vars, num_clauses, pos_lits, neg_lits = parse_cnf(file_path)
			
			if num_vars == 0 or num_clauses == 0:
				continue

			data = HeteroData()

			data['pos_lit'].x = self.lit_emb.expand(num_vars, self.hidden_channels)
			data['neg_lit'].x = self.lit_emb.expand(num_vars, self.hidden_channels)
			data['clause'].x = self.cls_emb.expand(num_clauses, self.hidden_channels)
			data['root'].x = torch.randn(1, self.hidden_channels)
			
			data['pos_lit', 'phs', 'neg_lit'].edge_index = torch.tensor(
				[[i for i in range(num_vars)], [i for i in range(num_vars)]], dtype=torch.long)
			data['pos_lit', 'or', 'clause'].edge_index = torch.tensor(pos_lits, dtype=torch.long)
			data['neg_lit', 'or', 'clause'].edge_index = torch.tensor(neg_lits, dtype=torch.long)
			data['clause', 'and', 'root'].edge_index = torch.tensor(
				[[i for i in range(num_clauses)], [0 for _ in range(num_clauses)]], dtype=torch.long)
				
			data =(data)

			# Store the label in the data object for tracking
			# We map string classes to integers (or just keep them as properties)
			# data.class_name = label 

			# Apply undirected transform
			data =  T.ToUndirected()(data)
			
			# Organize into our dictionary
			if label not in self.graphs_by_class:
				self.graphs_by_class[label] = []
				
			self.graphs_by_class[label].append(data)
			self.all_graphs.append(data)

	def len(self):
		# The number of steps in one epoch is usually the total number of graphs
		return 2 * self.num_bw # num bits in our dataset

	def get(self, idx):
		"""
		This is where the magic happens. Every time the dataloader asks for an item,
		we dynamically construct your required triplet.
		"""
		# if idx is even, we make anchor class as bhv_mul, else unr_mul
		classes = ['bhv_mul', 'unr_mul', 'rc_adr']
		anchor_cls_idx = idx % 2
		anchor_cls = classes[anchor_cls_idx]
		pos_cls = classes[1-anchor_cls_idx]
		neg_cls = 'rc_adr'

		bit_width_idx = idx // 2
		return self.graphs_by_class[anchor_cls][bit_width_idx], self.graphs_by_class[pos_cls][bit_width_idx], self.graphs_by_class[neg_cls][bit_width_idx], (anchor_cls_idx, bit_width_idx+1)
		

In [163]:
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv, HeteroConv
import torch.nn.functional as F

class CNFGraphModel(nn.Module):
	def __init__(self, hidden_channels, out_channels):
		super().__init__()
		
		# Base dictionary for message passing (forward and reverse edges)
		conv_dict = {
			('pos_lit', 'phs', 'neg_lit'): SAGEConv(hidden_channels, hidden_channels),
			('pos_lit', 'or', 'clause'): SAGEConv(hidden_channels, hidden_channels),
			('neg_lit', 'or', 'clause'): SAGEConv(hidden_channels, hidden_channels),
			('clause', 'and', 'root'): SAGEConv(hidden_channels, hidden_channels),
			
			('neg_lit', 'rev_phs', 'pos_lit'): SAGEConv(hidden_channels, hidden_channels),
			('clause', 'rev_or', 'pos_lit'): SAGEConv(hidden_channels, hidden_channels),
			('clause', 'rev_or', 'neg_lit'): SAGEConv(hidden_channels, hidden_channels),
			('root', 'rev_and', 'clause'): SAGEConv(hidden_channels, hidden_channels),
		}

		# Round 1
		self.conv1 = HeteroConv(conv_dict, aggr='mean')
		
		# Round 2
		conv_dict2 = {k: SAGEConv(hidden_channels, hidden_channels) for k in conv_dict.keys()}
		self.conv2 = HeteroConv(conv_dict2, aggr='mean')
		
		# Round 3
		conv_dict3 = {k: SAGEConv(hidden_channels, hidden_channels) for k in conv_dict.keys()}
		self.conv3 = HeteroConv(conv_dict3, aggr='mean')
		
		# Round 4
		conv_dict4 = {k: SAGEConv(hidden_channels, hidden_channels) for k in conv_dict.keys()}
		self.conv4 = HeteroConv(conv_dict4, aggr='mean')
		
		self.classifier = nn.Linear(hidden_channels, out_channels)

	def forward(self, x_dict, edge_index_dict, return_embeddings=False, iterations=2):
		# We loop through the entire block of layers
		for _ in range(iterations):
			# Pass 1
			x_dict = self.conv1(x_dict, edge_index_dict)
			x_dict = {key: F.relu(x) for key, x in x_dict.items()}
			
			# Pass 2
			x_dict = self.conv2(x_dict, edge_index_dict)
			x_dict = {key: F.relu(x) for key, x in x_dict.items()}
			
			# Pass 3
			x_dict = self.conv3(x_dict, edge_index_dict)
			x_dict = {key: F.relu(x) for key, x in x_dict.items()}
			
			# Pass 4
			x_dict = self.conv4(x_dict, edge_index_dict)
			
			# Optional: You can apply ReLU here, or a different activation 
			# like LayerNorm before it loops back to the start
			x_dict = {key: F.relu(x) for key, x in x_dict.items()} 
			
		# --- After all iterations are complete ---
		
		# Classification head
		root_embedding = x_dict['root']
		out = self.classifier(root_embedding)
		
		if return_embeddings:
			return out, x_dict
			
		return out

In [164]:
# --- RUNNING THE MODEL ---
# hidden_channels=32 to match your input features, out=1 for binary classification
model = CNFGraphModel(hidden_channels=64, out_channels=128)

# Forward pass example
# output = model(data.x_dict, data.edge_index_dict, True, 100)
# print("Output shape:", output) # Should be [1, 1]

In [165]:
# 1. Create your mapping (this could easily be read from a CSV file)
my_files = []
max_bw = 4
max_bw += 1
my_files.extend([(str(cnf_path) + f"/behavioral_mult_{i}bit.cnf", "bhv_mul") for i in range(1, max_bw)])
my_files.extend([(str(cnf_path) + f"/unrolled_mult_{i}bit.cnf", "unr_mul") for i in range(1, max_bw)])
my_files.extend([(str(cnf_path) + f"/rc_addr_{i}bit.cnf", "rc_adr") for i in range(1, max_bw)])

# 2. Initialize the dataset
dataset = TripletCNFDataset(file_class_mapping=my_files, hidden_channels=64, num_bw=4)

# 3. Create the DataLoader
# batch_size=16 means it will pull 16 triplets per step
loader = DataLoader(dataset, batch_size=28, shuffle=True)

triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

optimizer = torch.optim.Adam(list(model.parameters()) + [dataset.lit_emb, dataset.cls_emb], 3e-3)
# 4. Training Loop
for epoch in range(500):
	total_loss = 0
	iters = 0
	for step, (batch_1, batch_2, batch_3, meta) in enumerate(loader):
		iters += 1
		# batch_1 contains 16 graphs from your first class logic
		# batch_2 contains 16 graphs from your second class logic
		# batch_3 contains 16 graphs from your third class logic
		
		# print(f"Step {step}:")
		# print("  Batch 1 Root shape:", batch_1['root'].x.shape)
		# print("  Batch 2 Root shape:", batch_2['root'].x.shape)
		# print("  Batch 3 Root shape:", batch_3['root'].x.shape)
		
		# with torch.no_grad():
			# Run all three through your model to get their respective embeddings
		out_1 = model(batch_1.x_dict, batch_1.edge_index_dict)
		out_2 = model(batch_2.x_dict, batch_2.edge_index_dict)
		out_3 = model(batch_3.x_dict, batch_3.edge_index_dict)
		
		# Calculate your custom loss (e.g., TripletMarginLoss, Contrastive Loss, etc.)
		loss = triplet_loss(out_1, out_2, out_3)
		# loss.backward()
		# optimizer.step()
		total_loss += loss.item()

		# print(f"Meta: {meta} Loss: {loss.item()}")

	print("======================================================================")
	print(f"Loss at epoch {epoch+1} is {total_loss/iters}")
	print("======================================================================")

Processing CNF files into memory...
Loss at epoch 1 is 0.9990248680114746
Loss at epoch 2 is 0.9990249276161194
Loss at epoch 3 is 0.9990249872207642
Loss at epoch 4 is 0.9990249872207642
Loss at epoch 5 is 0.9990249276161194
Loss at epoch 6 is 0.9990249276161194
Loss at epoch 7 is 0.9990249276161194
Loss at epoch 8 is 0.9990249276161194
Loss at epoch 9 is 0.9990249872207642
Loss at epoch 10 is 0.9990249276161194
Loss at epoch 11 is 0.9990249872207642
Loss at epoch 12 is 0.9990249872207642
Loss at epoch 13 is 0.9990249872207642
Loss at epoch 14 is 0.9990249872207642
Loss at epoch 15 is 0.9990249872207642
Loss at epoch 16 is 0.9990249276161194
Loss at epoch 17 is 0.9990249872207642
Loss at epoch 18 is 0.9990249276161194
Loss at epoch 19 is 0.9990249276161194
Loss at epoch 20 is 0.9990249872207642
Loss at epoch 21 is 0.9990249276161194
Loss at epoch 22 is 0.9990249276161194
Loss at epoch 23 is 0.9990249872207642
Loss at epoch 24 is 0.9990249872207642
Loss at epoch 25 is 0.999024987220764

import torch

data = HeteroData()
data['pos_lit'].x = torch.randn(vars, 32)
data['neg_lit'].x = torch.randn(vars, 32)
data['clause'].x = torch.randn(clauses, 32)
data['root'].x = torch.randn(1, 32)


data['pos_lit', 'phs', 'neg_lit'].edge_index = torch.tensor([[i for i in range(vars)] for _ in range(2)], dtype=torch.long) 
data['pos_lit', 'or', 'clause'].edge_index = torch.tensor(pos_lits, dtype=torch.long)
data['neg_lit', 'or', 'clause'].edge_index = torch.tensor(neg_lits, dtype=torch.long)
data['clause', 'and', 'root'].edge_index = torch.tensor([[i for i in range(clauses)], [0 for _ in range(clauses)]], dtype=torch.long)

5 # This was given in PyG docs as a naive example 

import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv, to_hetero


class GNN(torch.nn.Module):
	def __init__(self, hidden_channels, out_channels):
		super().__init__()
		self.conv1 = SAGEConv((-1, -1), hidden_channels)
		self.conv2 = SAGEConv((-1, -1), out_channels)

	def forward(self, x, edge_index):
		x = self.conv1(x, edge_index).relu()
		x = self.conv2(x, edge_index)
		return x


model = GNN(hidden_channels=64, out_channels=64)
model = to_hetero(model, data.metadata(), aggr='sum')